# 02 · Synthetic scenarios: when does within-subject ranking (L_mul) help?

The simulator (`sat/data/simulate.py`) has a known data-generating process, so every
number can be compared with the **oracle** (the true CIFs). Each sweep varies one knob:

| sweep | knob | what it tests |
|---|---|---|
| `order_spread` | spread of per-event baseline scales | covariate-driven vs population-fixed event order |
| `shared` | share of risk that is common to all events | "shared disease pathways" |
| `frailty` | unobserved shared frailty variance | dependence covariates cannot explain |
| `censor` | fraction of unobserved event slots | robustness to censoring |
| `regime` | competing / semi-competing / non-competing | the general multi-event setting (§1.2) |
| `nonlin`, `events`, `n` (optional) | non-linearity, K, sample size | |

Models: oracle, Cox PH, and our model with L_PCH, L_PCH+L_mul, L_PCH+L_rank+L_mul.

## Options

In [ ]:
NAME = "02_synthetic_scenarios"

# Each sweep varies ONE simulator knob around the base scenario (sat/data/simulate.py).
SWEEPS = ["order_spread", "shared", "frailty", "censor", "regime"]   # core
# optional, more expensive: "nonlin", "events", "n"
RECIPES  = ["nllpch", "nllpch_event_ranking", "nllpch_sample_event_ranking"]
SEEDS    = [0, 1, 2]
WITH_COX = True          # cheap linear reference

# --- execution --------------------------------------------------------------
SMOKE_TEST   = False    # False = the real run. True: 2 epochs, 1 seed, reduced plan, only to check
                        # that everything runs (~10-30 min); smoke results go to a separate folder.
SMOKE_EPOCHS = 2
WORKERS      = None     # parallel runs; None = 2 on Kaggle (4 vCPU), 4 locally
TIME_BUDGET_MIN = 11 * 60   # stop launching new runs after this (Kaggle sessions end at 12 h)
INSTALL_DEPS = True     # Kaggle only
GIT_URL      = "https://github.com/Parsagh05/dmmst.git"   # code is cloned from here (nothing to upload)
CLEAN_UP_AT_END = True  # Kaggle: delete the working repo copy (models, caches) after the report

## Setup (identical in every notebook)

In [ ]:
# ---- 1. get the code --------------------------------------------------------------
import os, sys, json, shutil, subprocess, time, hashlib
from pathlib import Path

ON_KAGGLE = Path("/kaggle/working").is_dir()

def _is_repo(p):
    return (p / "sat" / "finetune.py").is_file() and (p / "conf").is_dir()

if ON_KAGGLE:
    WORK = Path("/kaggle/working")
    REPO = WORK / "dmmst"
    if not _is_repo(REPO):
        # prefer the code dataset over repo copies inside attached notebook outputs (stale code)
        cands = [f.parent.parent for f in Path("/kaggle/input").rglob("finetune.py")
                 if f.parent.name == "sat" and _is_repo(f.parent.parent)]
        cands.sort(key=lambda p: ("code" not in str(p).lower(), len(p.parts)))
        src = cands[0] if cands else None
        if src is not None:
            print("copying code from attached dataset:", src)
            shutil.copytree(src, REPO, ignore=shutil.ignore_patterns(
                ".git", ".venv", "model-hub", "__pycache__", "results", "tb_logs"))
        elif GIT_URL:
            subprocess.run(["git", "clone", "--depth", "1", GIT_URL, str(REPO)], check=True)
        else:
            raise RuntimeError("Attach the dmmst code dataset (notebooks/new/README.md, step 1) or set GIT_URL.")
    RESULTS_ROOT = WORK / "results"
else:
    REPO = Path.cwd().resolve()
    while not _is_repo(REPO) and REPO != REPO.parent:
        REPO = REPO.parent
    RESULTS_ROOT = REPO / "results" / "new"
assert _is_repo(REPO), REPO
os.chdir(REPO)
sys.path.insert(0, str(REPO))
CODE_FINGERPRINT = hashlib.md5(b"".join(
    p.read_bytes() for p in sorted((REPO / "sat").rglob("*.py")) + sorted((REPO / "conf").rglob("*.yaml"))
)).hexdigest()[:10]
print("repo:", REPO, "| code fingerprint:", CODE_FINGERPRINT)

In [ ]:
# ---- 2. dependencies (Kaggle) ------------------------------------------------------
# numpy is NEVER moved: pandas/pyarrow in the image are compiled against it.
if ON_KAGGLE and INSTALL_DEPS:
    import numpy as _np, torch
    _tv = tuple(int(x) for x in torch.__version__.split("+")[0].split(".")[:2])
    pkgs = ["transformers==4.50.0", "datasets==3.4.1", "tokenizers>=0.21,<0.22", "evaluate>=0.4.3",
            "accelerate>=1.4.0", "hydra-core>=1.3.2", "hydra-colorlog>=1.2.0",
            "torchsurv" if _tv >= (2, 8) else "torchsurv==0.1.4",
            "h5py>=3.13", "logdecorator>=2.5", "einops==0.8.1", "torchtuples>=0.2.2",
            "numba", "nvidia-ml-py", "polars", "tqdm", "tabulate",
            "scikit-survival" if int(_np.__version__.split(".")[0]) >= 2 else "scikit-survival==0.22.2",
            f"numpy=={_np.__version__}"]
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=True)
import torch
print("torch", torch.__version__, "| cuda" if torch.cuda.is_available() else "| cpu")

In [ ]:
# ---- 3. runner ----------------------------------------------------------------------
from IPython.display import Markdown, display
from scripts.runner import Run, Runner, dataset_overrides
from scripts.report import build as build_report

RUN_NAME = NAME + ("_smoke" if SMOKE_TEST else "")
r = Runner(repo=REPO, results_dir=RESULTS_ROOT / RUN_NAME,
           workers=WORKERS or (2 if ON_KAGGLE else 4),
           smoke_epochs=SMOKE_EPOCHS if SMOKE_TEST else None, gpu="0")

# Resume: attach this notebook's previous output ("Add Input" -> your notebook) and the
# finished runs are copied back, so only the missing ones are trained.
if ON_KAGGLE:
    n_copied = 0
    for d in Path("/kaggle/input").rglob(RUN_NAME):
        if d.is_dir() and d != r.results_dir:
            for f in d.glob("*.json"):
                if not (r.results_dir / f.name).exists():
                    shutil.copy2(f, r.results_dir / f.name); n_copied += 1
    print(f"resumed {n_copied} files from attached inputs")
(r.results_dir / "environment.json").write_text(json.dumps({
    "code_fingerprint": CODE_FINGERPRINT, "torch": torch.__version__,
    "cuda": torch.cuda.is_available(), "smoke": SMOKE_TEST, "name": RUN_NAME}, indent=2))
print("results ->", r.results_dir)

## Scenarios

In [ ]:
import dataclasses, re
from sat.data.simulate import SCENARIOS

KNOB = {"order_spread": "scale_spread", "shared": "shared_effect", "frailty": "frailty_var",
        "nonlin": "nonlinearity", "censor": "censor_rate", "events": "num_events", "n": "n",
        "regime": "regime"}
base = dataclasses.asdict(SCENARIOS["base"])
BASE_KNOBS = {sw: base[KNOB[sw]] for sw in KNOB}

def same_as_base(sc):
    d = dataclasses.asdict(sc); d.pop("name"); b = dict(base); b.pop("name")
    return d == b

plan = {}   # dataset -> (sweep, value)
for sw in SWEEPS:
    pat = "regime_" if sw == "regime" else rf"{sw}\d"      # "n" must not match "nonlin..."
    names = [n for n in SCENARIOS if re.match(pat, n)]
    if SMOKE_TEST:
        names = names[:2]
    for n in names:
        sc = SCENARIOS[n]
        if not same_as_base(sc):
            plan[f"sim_{n}"] = (sw, getattr(sc, KNOB[sw]))
plan["sim_base"] = ("base", None)
if SMOKE_TEST:
    SEEDS = SEEDS[:1]
print(len(plan), "datasets:", list(plan))

for ds in plan:
    if not (REPO / "data" / ds / "meta.json").is_file():
        subprocess.run([sys.executable, "-m", "sat.data.simulate", "--scenario", ds[4:], "--out", f"data/{ds}"],
                       check=True, stdout=subprocess.DEVNULL)
    r.prepare("multievent/survival", dataset_overrides(ds, REPO), label=ds)

## Plan and run

In [ ]:
runs = []
for s in SEEDS:
    so = [f"seed={s}", f"split_seed={s}"]
    for ds, (sw, val) in plan.items():
        dov = dataset_overrides(ds, REPO)
        base_info = {"dataset": ds, "seed": s, "sweep": sw, "value": val}
        if sw == "base":
            base_info["base_knobs"] = BASE_KNOBS
        runs.append(Run(f"{ds}__oracle__s{s}", "oracle", "multievent/survival", dov + so, ds, base_info | {"model": "oracle"}))
        if WITH_COX:
            runs.append(Run(f"{ds}__coxph__s{s}", "coxph", "multievent/coxph", dov + so, ds, base_info | {"model": "coxph"}))
        for rec in RECIPES:
            runs.append(Run(f"{ds}__ours_{rec}__s{s}", "finetune", "multievent/survival",
                            dov + so + [f"tasks/losses={rec}"], ds, base_info | {"model": "ours", "recipe": rec}))
print(len(runs), "runs")
r.run_many(runs, time_budget_min=TIME_BUDGET_MIN)

## Results
One figure per sweep (`report/sweep_<knob>.pdf|png`, table behind it in the `.csv`).
The question each sweep answers: **when does `L_mul` help?** Expected: the within-subject gain grows when the event order is driven by covariates (small `order_spread`) and shrinks when the population order already explains it (large `order_spread`), and depends on how much the events share (`shared`, `frailty`).

In [ ]:
# ---- report + zip -----------------------------------------------------------------------
df = build_report([r.results_dir], r.results_dir / "report")
rep = r.results_dir / "report" / "REPORT.md"
if rep.is_file():
    display(Markdown(rep.read_text(encoding="utf-8")))
print("zip:", r.zip())

from IPython.display import Image
for f in sorted((r.results_dir / "report").glob("sweep_*.png")):
    print(f.name); display(Image(filename=str(f)))

## Clean up
Keeps the Kaggle output small (results + zip only) and stops a later session from picking up this repo copy instead of the code dataset.

In [ ]:
if ON_KAGGLE and CLEAN_UP_AT_END:
    shutil.rmtree(REPO, ignore_errors=True)
    print("removed", REPO, "- results are in", r.results_dir)